# Temiz Model Egitimi - Varyant Patojenite Tahmini (v2)

**Amac**: Veri sizintisi (data leakage) giderilmis, Focal Loss entegreli, 4 FE modu ile kapsamli model karsilastirmasi.

| FE Modu | Aciklama | Sizinti |
|---------|----------|--------|
| no_fe | Legacy baseline (meta-predictor korunur) | EVET |
| with_fe | Legacy FE (tool consensus korunur) | EVET |
| **clean** | Temiz baseline (tum meta-predictor kaldirildi) | HAYIR |
| **clean_fe** | Temiz + gelismis FE (yonlu BLOSUM, PCA, delta k-mer) | HAYIR |

In [1]:
import sys, os
import warnings
import time

import pandas as pd
import numpy as np
import joblib
import torch

# Proje kokunu path'e ekle
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) != 'notebooks':
    PROJECT_ROOT = os.getcwd()
sys.path.insert(0, PROJECT_ROOT)

from config import *
from src.features import prepare_data, FE_PREPARERS
from src.models import MODEL_BUILDERS
from src.metrics import optimize_threshold, compute_all_metrics
from src.utils import get_train_test_data

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings('ignore')

print(f'Proje koku: {PROJECT_ROOT}')
print(f'FE modlari: {list(FE_PREPARERS.keys())}')
print(f'Model tipleri: {list(MODEL_BUILDERS.keys())}')
print(f'Optuna trial: Agac={TRIALS_TREE}, NN={TRIALS_NN}')

Proje koku: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model
FE modlari: ['no_fe', 'with_fe', 'clean', 'clean_fe']
Model tipleri: ['lightgbm', 'xgboost', 'nn', 'dnn']
Optuna trial: Agac=100, NN=50


c:\Users\ahmet.ceyhan23\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Veri yukle
df_raw = pd.read_csv(DATA_PATH)
print(f'Veri boyutu: {df_raw.shape}')
print(f'\nSinif dagilimi:')
print(df_raw['target'].value_counts())
print(f'\nPanel dagilimi:')
print(df_raw['Panel'].value_counts())

Veri boyutu: (4287, 119)

Sinif dagilimi:
target
0    2920
1    1367
Name: count, dtype: int64

Panel dagilimi:
Panel
General              3156
Hereditary_Cancer     715
PAH                   324
CFTR                   92
Name: count, dtype: int64


In [3]:
# Tum FE modlari icin veri hazirla
prepared_data = {}
for fe_mode in FE_PREPARERS.keys():
    print(f'\n{"="*60}')
    print(f'FE Modu: {fe_mode}')
    print('='*60)
    prepared_data[fe_mode] = prepare_data(df_raw.copy(), fe_mode)

print(f'\n{"="*60}')
print('OZET:')
for mode, df in prepared_data.items():
    print(f'  {mode:12s}: {df.shape[0]} x {df.shape[1]}')


FE Modu: no_fe
  DNA_11mer_Ref: 16 k-mer feature
  DNA_11mer_Alt: 16 k-mer feature
  Prot_11mer_Ref: 200 k-mer feature
  Prot_11mer_Alt: 200 k-mer feature
[No-FE] Veri boyutu: 4287 x 532

FE Modu: with_fe
  mutation_type: 12 one-hot feature
  aa_change: 150 one-hot feature
  r>0.99 kopya filtreleme: 4 sutun dusuruldu
[With-FE] Veri boyutu: 4287 x 217

FE Modu: clean
  98 sizintili/gereksiz sutun kaldirildi
  mutation_type: 12 one-hot feature
  aa_change: 150 one-hot feature
  DNA_11mer_Ref: 16 k-mer feature
  DNA_11mer_Alt: 16 k-mer feature
  Prot_11mer_Ref: 422 k-mer feature
  Prot_11mer_Alt: 425 k-mer feature
[Clean] Veri boyutu: 4287 x 1054

FE Modu: clean_fe
  Conservation PCA: 4 skor -> 2 PC (varyans: 97.69%)
  DNA_11mer_Ref/DNA_11mer_Alt: 16 k-mer + 16 delta
  Prot_11mer_Ref/Prot_11mer_Alt: 431 k-mer + 431 delta
  mutation_type: 12 one-hot feature
  aa_change: 150 one-hot feature
  98 sizintili/gereksiz sutun kaldirildi
  r>0.99 kopya filtreleme: 19 sutun dusuruldu
  Sifir varya

In [4]:
# DOGRULAMA: clean modlarda meta-predictor sutunu olmadigini kontrol et
from src.columns import LEAKY_META_PREDICTOR_SCORES, LEAKY_META_PREDICTOR_PREDS, RANKSCORE_COLS, GNOMAD_COLS

for mode in ['clean', 'clean_fe']:
    df_check = prepared_data[mode]
    leaked = [c for c in df_check.columns
              if c in LEAKY_META_PREDICTOR_SCORES + LEAKY_META_PREDICTOR_PREDS + RANKSCORE_COLS + GNOMAD_COLS]
    if leaked:
        print(f'HATA! {mode} modunda sizintili sutunlar bulundu: {leaked}')
    else:
        print(f'{mode}: Temiz -- hicbir sizintili sutun yok')

# Clean modlardaki sutunlari listele
print(f'\nClean-FE sutunlar ({len(prepared_data["clean_fe"].columns)}):')
for c in sorted(prepared_data['clean_fe'].columns):
    if c != 'target':
        print(f'  {c}')

clean: Temiz -- hicbir sizintili sutun yok
clean_fe: Temiz -- hicbir sizintili sutun yok

Clean-FE sutunlar (1497):
  DNA_11mer_Alt_aa
  DNA_11mer_Alt_ac
  DNA_11mer_Alt_ag
  DNA_11mer_Alt_at
  DNA_11mer_Alt_ca
  DNA_11mer_Alt_cc
  DNA_11mer_Alt_cg
  DNA_11mer_Alt_ct
  DNA_11mer_Alt_ga
  DNA_11mer_Alt_gc
  DNA_11mer_Alt_gg
  DNA_11mer_Alt_gt
  DNA_11mer_Alt_ta
  DNA_11mer_Alt_tc
  DNA_11mer_Alt_tg
  DNA_11mer_Alt_tt
  DNA_11mer_Ref_aa
  DNA_11mer_Ref_ac
  DNA_11mer_Ref_ag
  DNA_11mer_Ref_at
  DNA_11mer_Ref_ca
  DNA_11mer_Ref_cc
  DNA_11mer_Ref_cg
  DNA_11mer_Ref_ct
  DNA_11mer_Ref_ga
  DNA_11mer_Ref_gc
  DNA_11mer_Ref_gg
  DNA_11mer_Ref_gt
  DNA_11mer_Ref_ta
  DNA_11mer_Ref_tc
  DNA_11mer_Ref_tg
  DNA_11mer_Ref_tt
  Panel
  Prot_11mer_Alt__a
  Prot_11mer_Alt__c
  Prot_11mer_Alt__f
  Prot_11mer_Alt__g
  Prot_11mer_Alt__i
  Prot_11mer_Alt__k
  Prot_11mer_Alt__l
  Prot_11mer_Alt__s
  Prot_11mer_Alt__v
  Prot_11mer_Alt__w
  Prot_11mer_Alt_a_
  Prot_11mer_Alt_aa
  Prot_11mer_Alt_ac
  Prot_1

In [5]:
# Konfigurasyon listesi olustur
CONFIGS = []

for model_type in MODEL_TYPES:
    for fe_state in FE_MODES:
        # Multi-panel
        config_id = f'{model_type}_{fe_state}_multi_panel'
        CONFIGS.append({
            'config_id': config_id,
            'model_type': model_type,
            'fe_state': fe_state,
            'mode': 'multi_panel',
            'panel': None,
        })
        # Single-panel
        for panel in PANELS_SINGLE:
            config_id = f'{model_type}_{fe_state}_single_{panel}'
            CONFIGS.append({
                'config_id': config_id,
                'model_type': model_type,
                'fe_state': fe_state,
                'mode': 'single_panel',
                'panel': panel,
            })

print(f'Toplam konfigurasyon: {len(CONFIGS)}')
for c in CONFIGS[:8]:
    print(f'  {c["config_id"]}')

Toplam konfigurasyon: 64
  lightgbm_no_fe_multi_panel
  lightgbm_no_fe_single_General
  lightgbm_no_fe_single_Hereditary_Cancer
  lightgbm_no_fe_single_PAH
  lightgbm_with_fe_multi_panel
  lightgbm_with_fe_single_General
  lightgbm_with_fe_single_Hereditary_Cancer
  lightgbm_with_fe_single_PAH


In [6]:
from sklearn.metrics import f1_score as _f1_score

def run_single_config(config, prepared_data_dict, use_focal=False):
    """Tek bir konfigurasyonu bastan sona calistir."""
    config_id = config['config_id']
    model_type = config['model_type']
    fe_state = config['fe_state']
    mode = config['mode']
    panel = config.get('panel', None)

    df_prepared = prepared_data_dict[fe_state]

    X_train, X_test, y_train, y_test, cat_features = get_train_test_data(
        df_prepared, mode, panel
    )

    print(f'  Veri: {X_train.shape[0]} train, {X_test.shape[0]} test, {X_train.shape[1]} feature')

    builder = MODEL_BUILDERS[model_type]
    start_time = time.time()
    model, study, best_thr, y_pred_proba = builder(
        X_train, y_train, X_test, y_test, cat_features, use_focal=use_focal
    )
    elapsed = time.time() - start_time

    y_pred = (y_pred_proba >= best_thr).astype(int)
    metrics = compute_all_metrics(y_test, y_pred, y_pred_proba)

    y_pred_default = (y_pred_proba >= 0.5).astype(int)
    f1_default = _f1_score(y_test, y_pred_default, zero_division=0)

    result = {
        'config_id': config_id,
        'model_type': model_type,
        'fe_state': fe_state,
        'mode': mode,
        'panel': panel if panel else 'All',
        'best_threshold': best_thr,
        'f1_default': f1_default,
        'best_cv_f1': study.best_value if study else metrics['f1'],
        'n_train': X_train.shape[0],
        'n_test': X_test.shape[0],
        'n_features': X_train.shape[1],
        **metrics,
        'time_seconds': elapsed,
        '_model': model,
        '_study': study,
        '_y_test': y_test.values,
        '_y_pred_proba': y_pred_proba,
        '_y_pred': y_pred,
        '_feature_names': list(X_train.columns) if hasattr(X_train, 'columns') else None,
    }
    return result

In [7]:
# === ANA EGITIM DONGUSU ===
RESULTS = {}
total = len(CONFIGS)

# Focal Loss'u sadece clean modlarda kullan
USE_FOCAL_FOR_CLEAN = True

for i, config in enumerate(CONFIGS, 1):
    config_id = config['config_id']
    fe_state = config['fe_state']

    print(f'\n[{i}/{total}] {config_id}')
    print('-' * 50)

    use_focal = USE_FOCAL_FOR_CLEAN and fe_state in ['clean', 'clean_fe']

    try:
        result = run_single_config(config, prepared_data, use_focal=use_focal)
        RESULTS[config_id] = result
        print(f'  F1={result["f1"]:.4f} | AUC-ROC={result["auc_roc"]:.4f} | '
              f'Threshold={result["best_threshold"]:.2f} | {result["time_seconds"]:.1f}s')
    except Exception as e:
        print(f'  HATA: {e}')
        import traceback
        traceback.print_exc()

print(f'\n{"="*60}')
print(f'Tamamlanan konfigurasyon: {len(RESULTS)}/{total}')


[1/64] lightgbm_no_fe_multi_panel
--------------------------------------------------
  Veri: 3429 train, 858 test, 534 feature


Best trial: 60. Best value: 0.694307: 100%|██████████| 100/100 [14:50<00:00,  8.91s/it]


  F1=0.7097 | AUC-ROC=0.8719 | Threshold=0.53 | 892.6s

[2/64] lightgbm_no_fe_single_General
--------------------------------------------------
  Veri: 2524 train, 632 test, 530 feature


Best trial: 31. Best value: 0.672024: 100%|██████████| 100/100 [10:53<00:00,  6.54s/it]


  F1=0.6599 | AUC-ROC=0.8496 | Threshold=0.40 | 656.7s

[3/64] lightgbm_no_fe_single_Hereditary_Cancer
--------------------------------------------------
  Veri: 572 train, 143 test, 530 feature


...

  F1=0.5250 | AUC-ROC=0.6516 | Threshold=0.48 | 1557.9s

[63/64] dnn_clean_fe_single_Hereditary_Cancer
--------------------------------------------------
  Veri: 572 train, 143 test, 1495 feature


Best trial: 45. Best value: 0.568191: 100%|██████████| 50/50 [04:47<00:00,  5.75s/it]


  F1=0.6047 | AUC-ROC=0.7811 | Threshold=0.85 | 289.9s

[64/64] dnn_clean_fe_single_PAH
--------------------------------------------------
  Veri: 259 train, 65 test, 1495 feature


Best trial: 15. Best value: 0.533947: 100%|██████████| 50/50 [01:40<00:00,  2.01s/it]


  F1=0.5479 | AUC-ROC=0.6483 | Threshold=0.50 | 103.0s

Tamamlanan konfigurasyon: 64/64


In [8]:
# Sonuclari DataFrame'e cevir ve kaydet
csv_cols = ['config_id', 'model_type', 'fe_state', 'mode', 'panel',
            'best_threshold', 'f1_default', 'best_cv_f1',
            'n_train', 'n_test', 'n_features',
            'f1', 'auc_roc', 'auc_pr', 'mcc', 'precision', 'recall',
            'specificity', 'balanced_accuracy', 'cohens_kappa', 'time_seconds']

results_list = [{k: v for k, v in r.items() if not k.startswith('_')}
                for r in RESULTS.values()]
results_df = pd.DataFrame(results_list)[csv_cols]

os.makedirs(RESULTS_V2_DIR, exist_ok=True)
results_df.to_csv(os.path.join(RESULTS_V2_DIR, 'model_comparison_results.csv'), index=False)
print(f'Sonuclar kaydedildi: {RESULTS_V2_DIR}')

print(f'\nEN IYI 10 KONFIGURASYON (F1):')
print(results_df.nlargest(10, 'f1')[['config_id', 'f1', 'auc_roc', 'mcc', 'precision', 'recall']].to_string(index=False))

Sonuclar kaydedildi: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v2_clean

EN IYI 10 KONFIGURASYON (F1):
                                config_id       f1  auc_roc      mcc  precision   recall
 xgboost_with_fe_single_Hereditary_Cancer 0.814815 0.925296 0.742775   0.785714 0.846154
   xgboost_no_fe_single_Hereditary_Cancer 0.794872 0.931953 0.717949   0.794872 0.794872
  lightgbm_no_fe_single_Hereditary_Cancer 0.769231 0.928501 0.679528   0.673077 0.897436
lightgbm_with_fe_single_Hereditary_Cancer 0.717391 0.905819 0.602928   0.622642 0.846154
     dnn_with_fe_single_Hereditary_Cancer 0.712644 0.832347 0.595495   0.645833 0.794872
               lightgbm_no_fe_multi_panel 0.709677 0.871932 0.558046   0.635838 0.802920
             lightgbm_with_fe_multi_panel 0.707993 0.876919 0.556053   0.640118 0.791971
       dnn_no_fe_single_Hereditary_Cancer 0.707071 0.871055 0.592971   0.583333 0.897436
      nn_with_fe_single_Hereditary_Cancer 0.696629 0.876233 0.571744   0.620000 0.

In [9]:
# v1 (leaked) vs v2 (clean) karsilastirma
v1_path = os.path.join(RESULTS_V1_DIR, 'model_comparison_results.csv')
if os.path.exists(v1_path):
    v1_df = pd.read_csv(v1_path)
    v2_clean = results_df[results_df['fe_state'].isin(['clean', 'clean_fe'])]

    print('=' * 70)
    print('v1 (SIZINTILI) vs v2 (TEMIZ) KARSILASTIRMA')
    print('=' * 70)

    print('\nv1 (Leaked) - Model bazinda ortalama F1:')
    print(v1_df.groupby('model_type')['f1'].mean().sort_values(ascending=False).to_string())

    print('\nv2 Clean-FE - Model bazinda ortalama F1:')
    v2_cfe = results_df[results_df['fe_state'] == 'clean_fe']
    print(v2_cfe.groupby('model_type')['f1'].mean().sort_values(ascending=False).to_string())

    comparison = pd.DataFrame({
        'v1_best_f1': v1_df.groupby(['model_type', 'panel'])['f1'].max(),
        'v2_clean_fe_f1': results_df[results_df['fe_state']=='clean_fe'].groupby(['model_type', 'panel'])['f1'].max()
    }).reset_index()
    comparison['delta'] = comparison['v2_clean_fe_f1'] - comparison['v1_best_f1']
    comparison.to_csv(os.path.join(RESULTS_V2_DIR, 'v1_vs_v2_comparison.csv'), index=False)
    print('\nKarsilastirma tablosu:')
    print(comparison.to_string(index=False))
else:
    print('v1 sonuclari bulunamadi.')

v1 (SIZINTILI) vs v2 (TEMIZ) KARSILASTIRMA

v1 (Leaked) - Model bazinda ortalama F1:
model_type
lightgbm    0.675768
xgboost     0.666558
dnn         0.660391
nn          0.643120

v2 Clean-FE - Model bazinda ortalama F1:
model_type
lightgbm    0.603887
xgboost     0.570540
dnn         0.559837
nn          0.552113

Karsilastirma tablosu:
model_type             panel  v1_best_f1  v2_clean_fe_f1     delta
       dnn               All    0.685271        0.561713 -0.123558
       dnn           General    0.650602        0.525038 -0.125564
       dnn Hereditary_Cancer    0.755102        0.604651 -0.150451
       dnn               PAH    0.666667        0.547945 -0.118721
  lightgbm               All    0.702403        0.579017 -0.123386
  lightgbm           General    0.682809        0.612069 -0.070740
  lightgbm Hereditary_Cancer    0.790123        0.620690 -0.169434
  lightgbm               PAH    0.560000        0.603774  0.043774
        nn               All    0.679583        0.562712

In [14]:
# En iyi modelleri kaydet
import copy
os.makedirs(MODELS_V2_DIR, exist_ok=True)

def _safe_save_model(model, path):
    """Modeli pickle-safe sekilde kaydet."""
    if hasattr(model, 'state_dict'):
        # PyTorch modeli
        torch.save(model.state_dict(), path + '.pt')
        return path + '.pt'
    elif hasattr(model, 'save_model'):
        # XGBoost native save
        model.save_model(path + '.json')
        return path + '.json'
    else:
        # LightGBM vb. -- focal loss closure'u temizle
        m = copy.deepcopy(model)
        # LGBMClassifier icindeki pickle-edilemeyen objective'i kaldir
        if hasattr(m, '_objective') and callable(m._objective):
            m._objective = None
        if hasattr(m, 'objective') and callable(m.objective):
            m.objective = None
        # Booster icindeki custom objective
        if hasattr(m, '_Booster') and m._Booster is not None:
            pass  # Booster kendi icinde pickle-safe
        joblib.dump(m, path + '.joblib')
        return path + '.joblib'

for panel in ['All'] + PANELS_SINGLE:
    panel_results = {k: v for k, v in RESULTS.items()
                     if v['panel'] == panel and v['fe_state'] in ['clean', 'clean_fe']}
    if not panel_results:
        continue

    best_key = max(panel_results, key=lambda k: panel_results[k]['f1'])
    best = panel_results[best_key]

    model = best['_model']
    model_name = f'best_{panel}_{best["model_type"]}_{best["fe_state"]}'
    save_path = os.path.join(MODELS_V2_DIR, model_name)

    try:
        saved = _safe_save_model(model, save_path)
        print(f'Kaydedildi: {os.path.basename(saved)} (F1={best["f1"]:.4f})')
    except Exception as e:
        print(f'UYARI: {model_name} kaydedilemedi: {e}')

# Optuna study kaydet
os.makedirs(STUDIES_DIR, exist_ok=True)
for config_id, result in RESULTS.items():
    study = result.get('_study')
    if study is not None:
        try:
            joblib.dump(study, os.path.join(STUDIES_DIR, f'optuna_study_{config_id}.joblib'))
        except Exception as e:
            print(f'UYARI: Study {config_id} kaydedilemedi: {e}')

print('\nTum modeller ve study\'ler kaydedildi.')

Kaydedildi: best_All_xgboost_clean.json (F1=0.5982)
Kaydedildi: best_General_lightgbm_clean_fe.joblib (F1=0.6121)
Kaydedildi: best_Hereditary_Cancer_xgboost_clean_fe.json (F1=0.6526)
Kaydedildi: best_PAH_lightgbm_clean_fe.joblib (F1=0.6038)

Tum modeller ve study'ler kaydedildi.
